# IFEval: canonical, self-corrective, and LANL ensemble

IFEval contains 541 instruction-following prompts with deterministic checks for requirements such
as word counts, required sections, and forbidden punctuation. Grading uses the vendored official
verifier and makes no grading-model calls.

This notebook compares three independently revisioned Engine protocols:

- `ifeval` invokes the complete Candidate once and grades its final answer.
- `ifeval/self-corrective` gives deterministic failure feedback to the complete Candidate for up
  to three attempts.
- `ifeval/lanl-ensemble` invokes each direct Fusion member, stops when a member passes, and uses
  the configured synthesizer route only for benchmark-owned coaching or exact tie-breaking.

Every evaluation cell below performs paid Candidate calls. Start with `limit=1`, inspect the
Reports, and increase the selection deliberately.

## Before running

From a terminal in `packages/screamingface/`:

```bash
just stack-prepare  # first run only: download pinned Benchmark assets
just stack-up       # start AI Gateway :9105 and Engine :9108
just stack-status
```

Use `just stack-logs` to inspect startup failures and `just stack-down` when finished. Stack
management stays outside the notebook so **Run All** never starts or stops local services.

In [ ]:
import screamingface as sf

sf.connect()

## Define Candidate-owned answer and synthesis policy

An explicit synthesizer `sf.Model` makes its whole-Fusion prompt and generation parameters
visible. The LANL protocol keeps that Model's route and parameters but replaces the ordinary
blending prompt with its own revisioned coaching and selection instructions.

In [ ]:
ANSWER_PROMPT = (
    "Answer the request accurately and completely. "
    "Follow every instruction and formatting constraint in the request."
)
SYNTHESIS_PROMPT = (
    "Produce one final answer to the original request from the panel drafts. "
    "Preserve every instruction and formatting constraint."
)

haiku = sf.Model(
    "openrouter/anthropic/claude-haiku-4.5",
    prompt=ANSWER_PROMPT,
    params={"max_tokens": 4096},
)
gemini = sf.Model(
    "openrouter/google/gemini-3-flash-preview",
    prompt=ANSWER_PROMPT,
    params={"max_tokens": 4096},
)
kimi = sf.Model(
    "openrouter/moonshotai/kimi-k2.6",
    prompt=ANSWER_PROMPT,
    params={"max_tokens": 4096},
)
panel = sf.Fusion(
    [haiku, gemini],
    name="haiku+gemini",
    synthesizer=sf.Model(
        "openrouter/moonshotai/kimi-k2.6",
        prompt=SYNTHESIS_PROMPT,
        params={"max_tokens": 4096},
    ),
)
[kimi, panel]

## 1. Canonical solo baseline

In [ ]:
canonical_solo = sf.evaluate(kimi, benchmark="ifeval", limit=1)
canonical_solo

## 2. Canonical whole-Fusion synthesis

In [ ]:
canonical_fusion = sf.evaluate(panel, benchmark="ifeval", limit=1)
canonical_fusion

## 3. Self-corrective Candidate

In [ ]:
self_corrective = sf.evaluate(
    sf.Model(
        "openrouter/moonshotai/kimi-k2.6",
        prompt=ANSWER_PROMPT,
        params={"max_tokens": 16384},
    ),
    benchmark="ifeval/self-corrective",
    limit=1,
)
self_corrective

## 4. LANL early-exit ensemble

In [ ]:
lanl_ensemble = sf.evaluate(
    panel,
    benchmark="ifeval/lanl-ensemble",
    limit=1,
)
lanl_ensemble

## Compare complete portable artifacts

In [ ]:
{
    "canonical_solo": canonical_solo.to_dict(),
    "canonical_fusion": canonical_fusion.to_dict(),
    "self_corrective": self_corrective.to_dict(),
    "lanl_ensemble": lanl_ensemble.to_dict(),
}